# LeetCode 673 — Number of Longest Increasing Subsequence bằng Python

Link đề: https://leetcode.com/problems/number-of-longest-increasing-subsequence/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Longest Increasing Subsequence, Binary Indexed Tree, Segment Tree

Skeleton LeetCode:

```python
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        ...
```

Ghi chú: bài không cần cấu trúc tự dựng. Đây là phiên bản "đếm" của bài kinh điển **300. Longest Increasing Subsequence**: không chỉ hỏi độ dài dãy con tăng dài nhất mà hỏi **có bao nhiêu dãy** như vậy. Notebook đi qua 5 cách: vét cạn mọi dãy con (O(2^n)), DP O(n²) với hai mảng `f` và `cnt`, DFS có nhớ (hướng "bắt đầu tại i"), Fenwick tree (BIT) O(n log n), và "xếp tầng + tổng tiền tố" O(n log n).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: đếm số **dãy con tăng ngặt** có **độ dài lớn nhất** của mảng `nums` (hai dãy con khác nhau nếu chọn **khác tập chỉ số**, kể cả khi giá trị giống nhau).
2. Nắm các công cụ Python: **dãy con** (subsequence) khác **mảng con** (subarray), `itertools.combinations`, vòng lặp lồng `for i ... for j in range(i)`, cặp `(độ dài, số cách)` và cách **gộp** hai cặp, `bisect`, cùng cây chỉ số nhị phân (Fenwick / BIT).
3. Đi từ **liệt kê 2^n dãy con** → **DP hai mảng `f[i]`, `cnt[i]`** → **DFS có nhớ theo hướng ngược** → **Fenwick tree O(n log n)** → **xếp tầng (patience) với tổng tiền tố đếm**.
4. Hiểu mấu chốt: mỗi vị trí `i` giữ **cặp** `(độ dài LIS kết thúc tại i, số dãy đạt độ dài đó)`; khi nối thêm `nums[i]` vào các `j` với `nums[j] < nums[i]` thì **độ dài lấy max, số cách cộng dồn khi bằng nhau** (nếu độ dài mới lớn hơn thì **đặt lại** số cách).
5. Tránh các lỗi kinh điển: dùng `<=` thay `<` (không còn "tăng ngặt"), quên **đặt lại** `cnt[i]` khi tìm được độ dài tốt hơn, khởi tạo `cnt = 0`, trả về `cnt[-1]` thay vì **cộng** `cnt` mọi vị trí đạt độ dài lớn nhất, và nhầm với bài tìm **độ dài** LIS.

## 2. Đề bài & ví dụ

Cho một mảng số nguyên `nums`. Hãy trả về **số lượng dãy con tăng dài nhất**.

Chú ý: dãy phải **tăng ngặt** (strictly increasing): phần tử sau **lớn hơn hẳn** phần tử trước.

**Ví dụ 1**

- Input: `nums = [1,3,5,4,7]`
- Output: `2`
- Giải thích: hai dãy con tăng dài nhất (độ dài 4) là `[1,3,4,7]` và `[1,3,5,7]`.

**Ví dụ 2**

- Input: `nums = [2,2,2,2,2]`
- Output: `5`
- Giải thích: độ dài dãy tăng ngặt dài nhất là `1`, và có `5` dãy độ dài 1 (mỗi phần tử là một dãy, khác nhau vì khác **vị trí**).

**Ràng buộc:**

- `1 <= nums.length <= 2000`
- `-10^6 <= nums[i] <= 10^6`
- Đáp án chắc chắn nằm trong 32-bit số nguyên có dấu.

Nhớ: **dãy con** giữ nguyên thứ tự nhưng **không cần liền kề** (khác mảng con). Ví dụ từ `[1,3,5,4,7]` ta có thể chọn các chỉ số `0,1,3,4` để được `[1,3,4,7]`.

In [1]:
from typing import List
from itertools import combinations
from bisect import bisect_left
import random
import sys
import time

examples = [
    ([1, 3, 5, 4, 7], 2),
    ([2, 2, 2, 2, 2], 5),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 3, 5, 4, 7] -> kỳ vọng 2
[2, 2, 2, 2, 2] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và `itertools.combinations`

Một dãy con được xác định bởi **tập chỉ số** `i1 < i2 < ... < ik` đã chọn. Mảng `n` phần tử có `2^n` dãy con (mỗi phần tử chọn hoặc không). `combinations(range(n), k)` liệt kê mọi tập `k` chỉ số tăng dần — đúng bằng mọi dãy con độ dài `k`.

In [2]:
nums = [1, 3, 5, 4]
for idx in combinations(range(len(nums)), 2):
    print("chỉ số", idx, "-> dãy", [nums[i] for i in idx])
print("tổng số dãy con (kể cả rỗng):", 2 ** len(nums))

chỉ số (0, 1) -> dãy [1, 3]
chỉ số (0, 2) -> dãy [1, 5]
chỉ số (0, 3) -> dãy [1, 4]
chỉ số (1, 2) -> dãy [3, 5]
chỉ số (1, 3) -> dãy [3, 4]
chỉ số (2, 3) -> dãy [5, 4]
tổng số dãy con (kể cả rỗng): 16


### 3.2. Tăng ngặt: so sánh mọi cặp kề nhau

Một dãy `a` là tăng ngặt nếu `a[k] < a[k+1]` với mọi `k`. Dùng `zip(a, a[1:])` để ghép các cặp kề nhau và `all(...)` để kiểm tra mọi cặp. Dãy 0 hoặc 1 phần tử tăng ngặt theo quy ước (không có cặp nào vi phạm).

In [3]:
def is_strictly_increasing(a):
    return all(x < y for x, y in zip(a, a[1:]))

print(is_strictly_increasing([1, 3, 4, 7]))
print(is_strictly_increasing([1, 3, 3, 7]))   # có hai số bằng nhau -> không ngặt
print(is_strictly_increasing([5]))            # một phần tử
print(is_strictly_increasing([]))             # rỗng

True
False
True
True


### 3.3. Vòng lặp lồng `for i ... for j in range(i)`

Để xét mọi cặp `(j, i)` với `j < i` ta viết `for i in range(n): for j in range(i):` — với mỗi `i`, `j` chạy qua `0 .. i-1`. Đây là dạng quen thuộc của DP "nhìn lại phía trước".

In [4]:
n = 4
for i in range(n):
    print(f"i={i}: j chạy qua", list(range(i)))

i=0: j chạy qua []
i=1: j chạy qua [0]
i=2: j chạy qua [0, 1]
i=3: j chạy qua [0, 1, 2]


### 3.4. Cặp `(độ dài, số cách)` và phép gộp

Ta sẽ phải nhớ **hai** con số cho mỗi bài toán con: độ dài tốt nhất `L` và số cách `C` đạt độ dài đó. Khi có **hai ứng viên** `(L1, C1)` và `(L2, C2)`:

- `L1 > L2` → giữ `(L1, C1)` (ứng viên kém hơn bị bỏ, số cách của nó **không** được cộng).
- `L1 == L2` → `(L1, C1 + C2)` (cùng độ dài tốt nhất, số cách cộng lại).
- `L1 < L2` → giữ `(L2, C2)`.

Phép gộp này giao hoán và kết hợp, nên có thể gộp theo thứ tự nào cũng được — điều quan trọng cho các cách dùng BIT.

In [5]:
def merge(a, b):
    """Gộp hai cặp (độ dài, số cách)."""
    if a[0] > b[0]:
        return a
    if a[0] < b[0]:
        return b
    return (a[0], a[1] + b[1])

print(merge((3, 2), (2, 5)))    # (3, 2): độ dài 3 thắng, bỏ số cách của (2, 5)
print(merge((3, 2), (3, 4)))    # (3, 6): bằng nhau -> cộng
print(merge((1, 1), (4, 7)))    # (4, 7)
from functools import reduce
print(reduce(merge, [(2, 1), (3, 1), (3, 2), (1, 9)]))   # (3, 3)

(3, 2)
(3, 6)
(4, 7)
(3, 3)


### 3.5. `bisect_left` — tìm vị trí chèn bằng nhị phân

`bisect_left(a, x)` trên list **đã sắp tăng** trả về vị trí đầu tiên `i` mà `a[i] >= x` (O(log n)). Ta dùng nó ở cách "xếp tầng". Với list **giảm dần** phải tự viết nhị phân hoặc đảo dấu.

In [6]:
a = [1, 3, 3, 5, 8]
print("bisect_left(a, 3) =", bisect_left(a, 3))   # 1: đứng trước các số 3
print("bisect_left(a, 4) =", bisect_left(a, 4))   # 3
print("bisect_left(a, 9) =", bisect_left(a, 9))   # 5: cuối list

bisect_left(a, 3) = 1
bisect_left(a, 4) = 3
bisect_left(a, 9) = 5


### 3.6. Cây chỉ số nhị phân (Fenwick tree / BIT)

BIT lưu thông tin trên các "đoạn" có độ dài là lũy thừa của 2 để hai thao tác **cập nhật một vị trí** và **truy vấn tiền tố** đều chỉ tốn O(log n). Với chỉ số bắt đầu từ `1`:

- Truy vấn tiền tố `[1..p]`: `while p > 0: dùng node p; p -= p & -p`.
- Cập nhật vị trí `p`: `while p <= m: cập nhật node p; p += p & -p`.

`p & -p` là bit `1` thấp nhất của `p`. Hãy xem các node mà mỗi thao tác đi qua với `m = 8`.

In [7]:
m = 8
for p in (5, 8, 3):
    q, path = p, []
    while q > 0:
        path.append(q)
        q -= q & -q
    print(f"truy vấn tiền tố {p}: đi qua node {path}")
for p in (1, 3, 6):
    q, path = p, []
    while q <= m:
        path.append(q)
        q += q & -q
    print(f"cập nhật vị trí {p}: đi qua node {path}")

truy vấn tiền tố 5: đi qua node [5, 4]
truy vấn tiền tố 8: đi qua node [8]
truy vấn tiền tố 3: đi qua node [3, 2]
cập nhật vị trí 1: đi qua node [1, 2, 4, 8]
cập nhật vị trí 3: đi qua node [3, 4, 8]
cập nhật vị trí 6: đi qua node [6, 8]


## 4. Ý tưởng nền tảng / trực giác

Với `nums = [1,3,5,4,7]`, hãy liệt kê tay các dãy con tăng ngặt kết thúc bằng `7`:

- Nối sau `[1,3,5]` → `[1,3,5,7]` (độ dài 4)
- Nối sau `[1,3,4]` → `[1,3,4,7]` (độ dài 4)
- Các dãy ngắn hơn như `[1,7]`, `[3,7]`... không dài nhất.

Ta thấy: để biết có bao nhiêu dãy dài nhất **kết thúc tại `7`**, ta chỉ cần biết, với **từng số đứng trước nhỏ hơn 7**, độ dài tốt nhất và số cách kết thúc tại số đó — rồi lấy những số cho độ dài **lớn nhất** và cộng số cách của chúng. Đó là tinh thần DP.

Trước khi tối ưu, viết một "trọng tài" cực chậm nhưng **chắc chắn đúng**: sinh mọi dãy con theo chỉ số, giữ dãy tăng ngặt, tìm độ dài lớn nhất và đếm.

In [8]:
def oracle(nums):
    """Trọng tài: duyệt mọi tập chỉ số (2^n). Chỉ dùng cho n nhỏ (<= ~14)."""
    n = len(nums)
    best, cnt = 0, 0
    for mask in range(1, 1 << n):
        seq = [nums[i] for i in range(n) if mask >> i & 1]
        if is_strictly_increasing(seq):
            if len(seq) > best:
                best, cnt = len(seq), 1
            elif len(seq) == best:
                cnt += 1
    return cnt

for nums, expected in examples:
    assert oracle(nums) == expected
    print(nums, "->", oracle(nums))
print(oracle([1, 3, 2, 4, 3, 5]))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5
3


Lưu ý: `oracle` dùng **bitmask** `mask` để biểu diễn tập chỉ số đã chọn (`mask >> i & 1` là bit thứ `i`). Mỗi `mask` từ `1` đến `2^n - 1` là một dãy con không rỗng.

## 5. Cách 1 — Brute Force: liệt kê mọi dãy con (O(2^n · n))

**Ý tưởng.** Chính là `oracle` ở mục 4, nhưng ta viết lại bằng đệ quy "chọn / không chọn" để thấy rõ cấu trúc: tại chỉ số `i`, hoặc **bỏ qua** `nums[i]`, hoặc **chọn** nó nếu lớn hơn phần tử đã chọn cuối. Mỗi lần dãy kết thúc ta cập nhật `best` và số đếm.

Sau đó ta sẽ thấy nó bùng nổ theo `2^n` — với `n = 2000` hoàn toàn bất khả thi.

In [9]:
def brute_force(nums):
    n = len(nums)
    best = 0
    count = 0

    def go(i, last, length):
        nonlocal best, count
        if i == n:
            if length > 0:
                if length > best:
                    best, count = length, 1
                elif length == best:
                    count += 1
            return
        go(i + 1, last, length)                    # không chọn nums[i]
        if length == 0 or nums[i] > last:          # chọn nums[i] nếu còn tăng ngặt
            go(i + 1, nums[i], length + 1)

    go(0, None, 0)
    return count

for nums, expected in examples:
    assert brute_force(nums) == expected
    print(nums, "->", brute_force(nums))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5


**Độ phức tạp.** Có tới `2^n` nhánh → **Time O(2^n)** (nhân thêm hằng số nhỏ), **Space O(n)** cho ngăn xếp đệ quy. `n = 30` đã là ~10^9 nhánh.

**Nút thắt.** Rất nhiều nhánh **lặp lại cùng một bài toán con**: "dãy tăng dài nhất kết thúc ở chỉ số `i`" được tính lại cho từng cách chọn tiền tố. Ta chỉ cần lưu, **cho mỗi `i`**, hai số: độ dài tốt nhất và số cách. Đó là bước đột phá.

## 6. Ý tưởng thuật toán / bước đột phá

Định nghĩa với mỗi vị trí `i`:

- `f[i]` = độ dài của dãy tăng ngặt dài nhất **kết thúc tại `nums[i]`**.
- `cnt[i]` = **số** dãy tăng ngặt có độ dài `f[i]` kết thúc tại `nums[i]`.

Ban đầu `f[i] = 1`, `cnt[i] = 1` (dãy chỉ gồm `nums[i]`).

Với mỗi `j < i` thỏa `nums[j] < nums[i]`, ta có thể nối `nums[i]` sau mọi dãy kết thúc tại `j`, được ứng viên `(f[j] + 1, cnt[j])`:

- nếu `f[j] + 1 > f[i]`: tìm được độ dài **tốt hơn** → `f[i] = f[j] + 1` và **đặt lại** `cnt[i] = cnt[j]`;
- nếu `f[j] + 1 == f[i]`: có thêm cách đạt cùng độ dài → `cnt[i] += cnt[j]`;
- nếu nhỏ hơn: bỏ qua.

Đáp án: gọi `mx = max(f)`, trả `sum(cnt[i] for i if f[i] == mx)`. (Mọi dãy dài nhất đều kết thúc tại **đúng một** `i`, nên không đếm trùng.)

Hãy chạy ý tưởng này trên ví dụ 1 và nhìn hai mảng.

In [10]:
nums = [1, 3, 5, 4, 7]
n = len(nums)
f = [1] * n
cnt = [1] * n
for i in range(n):
    for j in range(i):
        if nums[j] < nums[i]:
            if f[i] < f[j] + 1:
                f[i] = f[j] + 1
                cnt[i] = cnt[j]
            elif f[i] == f[j] + 1:
                cnt[i] += cnt[j]
print("nums =", nums)
print("f    =", f)
print("cnt  =", cnt)
mx = max(f)
print("mx =", mx, "-> đáp án =", sum(c for fi, c in zip(f, cnt) if fi == mx))

nums = [1, 3, 5, 4, 7]
f    = [1, 2, 3, 3, 4]
cnt  = [1, 1, 1, 1, 2]
mx = 4 -> đáp án = 2


Ở `i = 4` (số `7`): từ `j = 2` (số `5`, `f = 3`) ta được `f[4] = 4`, `cnt[4] = 1`; từ `j = 3` (số `4`, `f = 3`) cũng được độ dài `4` nên `cnt[4] += 1 = 2`. Đúng hai dãy `[1,3,5,7]` và `[1,3,4,7]`.

## 7. Các cách tiếp theo

5 cách tổng cộng (cách 1 đã ở mục 5):

| # | Cách | Time | Space |
|---|---|---|---|
| 1 | Liệt kê mọi dãy con | O(2^n) | O(n) |
| 2 | DP hai mảng `f`, `cnt` | O(n²) | O(n) |
| 3 | DFS có nhớ (hướng "bắt đầu tại i") | O(n²) | O(n) |
| 4 | Fenwick tree (BIT) trên giá trị | O(n log n) | O(n) |
| 5 | Xếp tầng + tổng tiền tố đếm | O(n log n) | O(n) |

### 7.1. Cách 2 — DP O(n²) với `f` và `cnt`

**Giải thích từng đoạn.**

1. `f = [1]*n`, `cnt = [1]*n`: mỗi phần tử tự nó là một dãy độ dài 1.
2. Hai vòng lặp `for i`, `for j in range(i)`: với mỗi cặp `j < i` mà `nums[j] < nums[i]` (**ngặt**).
3. Nếu `f[i] < f[j] + 1`: độ dài tốt hơn → cập nhật `f[i]` **và đặt lại** `cnt[i] = cnt[j]`.
4. Nếu `f[i] == f[j] + 1`: cùng độ dài → `cnt[i] += cnt[j]`.
5. Giữ thêm biến chạy `mx` (độ dài LIS toàn cục) và `ans` (số cách): sau khi xong `i`, nếu `f[i] > mx` thì `mx, ans = f[i], cnt[i]`; nếu `f[i] == mx` thì `ans += cnt[i]`.

Cách giữ `mx`, `ans` chạy cùng lúc cho phép khỏi phải quét lại mảng lần cuối.

In [11]:
def dp_quadratic(nums):
    n = len(nums)
    f = [1] * n
    cnt = [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i] = f[j] + 1
                    cnt[i] = cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

for nums, expected in examples:
    assert dp_quadratic(nums) == expected
    print(nums, "->", dp_quadratic(nums))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5


In [12]:
# Code nộp LeetCode — Cách 2: DP O(n^2)
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        n = len(nums)
        f = [1] * n
        cnt = [1] * n
        mx = ans = 0
        for i in range(n):
            for j in range(i):
                if nums[j] < nums[i]:
                    if f[i] < f[j] + 1:
                        f[i] = f[j] + 1
                        cnt[i] = cnt[j]
                    elif f[i] == f[j] + 1:
                        cnt[i] += cnt[j]
            if f[i] > mx:
                mx, ans = f[i], cnt[i]
            elif f[i] == mx:
                ans += cnt[i]
        return ans

for nums, expected in examples:
    assert Solution().findNumberOfLIS(nums) == expected
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp.** Hai vòng lồng nhau: **Time O(n²)** (`n = 2000` ≈ 2 triệu bước, nhanh). **Space O(n)** cho `f`, `cnt`.

### 7.2. Cách 3 — DFS có nhớ, hướng "bắt đầu tại i"

**Ý tưởng khác hướng.** Thay vì "kết thúc tại `i`" (nhìn lui), ta định nghĩa `dfs(i)` = cặp `(độ dài, số cách)` của dãy tăng dài nhất **bắt đầu tại `i`**. Với mỗi `j > i` mà `nums[j] > nums[i]`, ứng viên là `(L_j + 1, C_j)` từ `dfs(j)`; gộp bằng phép `merge` ở mục 3.4 (mặc định `(1, 1)` cho dãy chỉ có `nums[i]`). Rồi gộp `dfs(i)` qua mọi `i` để ra đáp án.

**Điểm cần chú ý:** đệ quy sâu tới độ dài LIS (tối đa 2000 khi mảng tăng hẳn) — vượt giới hạn mặc định 1000 của Python, nên ta **nâng** giới hạn (`sys.setrecursionlimit`) trong hàm. Dùng `dict` `memo` để mỗi `i` chỉ tính một lần.

In [13]:
def memo_dfs(nums):
    n = len(nums)
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, n + 100))
    memo = {}

    def dfs(i):
        if i in memo:
            return memo[i]
        best = (1, 1)                                  # chỉ riêng nums[i]
        for j in range(i + 1, n):
            if nums[j] > nums[i]:
                L, C = dfs(j)
                best = merge(best, (L + 1, C))
        memo[i] = best
        return best

    try:
        total = (0, 0)
        for i in range(n):
            total = merge(total, dfs(i))
        return total[1]
    finally:
        sys.setrecursionlimit(old)

for nums, expected in examples:
    assert memo_dfs(nums) == expected
    print(nums, "->", memo_dfs(nums))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5


In [14]:
# Code nộp LeetCode — Cách 3: DFS có nhớ (bắt đầu tại i)
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        import sys
        n = len(nums)
        sys.setrecursionlimit(max(sys.getrecursionlimit(), n + 100))
        memo = {}

        def merge(a, b):
            if a[0] != b[0]:
                return a if a[0] > b[0] else b
            return (a[0], a[1] + b[1])

        def dfs(i):
            if i in memo:
                return memo[i]
            best = (1, 1)
            for j in range(i + 1, n):
                if nums[j] > nums[i]:
                    L, C = dfs(j)
                    best = merge(best, (L + 1, C))
            memo[i] = best
            return best

        total = (0, 0)
        for i in range(n):
            total = merge(total, dfs(i))
        return total[1]

for nums, expected in examples:
    assert Solution().findNumberOfLIS(nums) == expected
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


**Độ phức tạp.** Có `n` trạng thái, mỗi trạng thái duyệt tới `n` ứng viên: **Time O(n²)**. **Space O(n)** cho `memo` và ngăn xếp đệ quy. Ưu điểm: dễ chuyển từ cách vét cạn bằng cách thêm `memo`; nhược điểm: phụ thuộc giới hạn đệ quy.

### 7.3. Cách 4 — Fenwick tree (BIT) trên giá trị, O(n log n)

**Ý tưởng.** Ở DP O(n²), phần tốn kém là "với mọi `j < i` có `nums[j] < nums[i]` hãy lấy cặp `(f, cnt)` tốt nhất rồi gộp". Đó chính là **truy vấn tiền tố trên trục giá trị**: gộp mọi cặp đã lưu ở các giá trị **nhỏ hơn** `nums[i]`. Phép `merge` là giao hoán, kết hợp → BIT dùng được.

**Giải thích từng đoạn.**

1. **Nén tọa độ**: `vals = sorted(set(nums))`, `idx[v]` = thứ hạng (từ 1). Giá trị có thể âm / rất lớn nên không dùng trực tiếp làm chỉ số.
2. BIT lưu hai mảng `tl[p]` (độ dài) và `tc[p]` (số cách) cho mỗi node.
3. Duyệt `x` theo thứ tự xuất hiện: `(L, C) = query(idx[x] - 1)` — gộp mọi giá trị **nhỏ hơn** `x`. Nếu `L == 0` (không có gì) thì `C = 1`. Dãy kết thúc tại `x` có `(L + 1, C)`.
4. `update(idx[x], L + 1, C)`: gộp cặp mới vào các node bao phủ `idx[x]`.
5. Đồng thời gộp `(L + 1, C)` vào đáp án toàn cục.

Nhiều phần tử **trùng giá trị** cùng dồn vào một vị trí BIT: phép gộp tự cộng số cách khi độ dài bằng nhau, và giữ cặp tốt hơn khi độ dài khác nhau — đúng ý nghĩa.

In [15]:
def bit_way(nums):
    vals = sorted(set(nums))
    idx = {v: k + 1 for k, v in enumerate(vals)}
    m = len(vals)
    tl = [0] * (m + 1)
    tc = [0] * (m + 1)

    def query(p):                       # gộp các vị trí 1..p
        bl, bc = 0, 0
        while p > 0:
            if tl[p] > bl:
                bl, bc = tl[p], tc[p]
            elif tl[p] == bl:
                bc += tc[p]
            p -= p & -p
        return bl, bc

    def update(p, l, c):
        while p <= m:
            if tl[p] < l:
                tl[p], tc[p] = l, c
            elif tl[p] == l:
                tc[p] += c
            p += p & -p

    best, total = 0, 0
    for x in nums:
        l, c = query(idx[x] - 1)
        if l == 0:
            c = 1
        l += 1
        update(idx[x], l, c)
        if l > best:
            best, total = l, c
        elif l == best:
            total += c
    return total

for nums, expected in examples:
    assert bit_way(nums) == expected
    print(nums, "->", bit_way(nums))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5


In [16]:
# Code nộp LeetCode — Cách 4: Fenwick tree
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        vals = sorted(set(nums))
        idx = {v: k + 1 for k, v in enumerate(vals)}
        m = len(vals)
        tl = [0] * (m + 1)
        tc = [0] * (m + 1)

        def query(p):
            bl, bc = 0, 0
            while p > 0:
                if tl[p] > bl:
                    bl, bc = tl[p], tc[p]
                elif tl[p] == bl:
                    bc += tc[p]
                p -= p & -p
            return bl, bc

        def update(p, l, c):
            while p <= m:
                if tl[p] < l:
                    tl[p], tc[p] = l, c
                elif tl[p] == l:
                    tc[p] += c
                p += p & -p

        best, total = 0, 0
        for x in nums:
            l, c = query(idx[x] - 1)
            if l == 0:
                c = 1
            l += 1
            update(idx[x], l, c)
            if l > best:
                best, total = l, c
            elif l == best:
                total += c
        return total

for nums, expected in examples:
    assert Solution().findNumberOfLIS(nums) == expected
print("Cách 4 qua ví dụ đề")

Cách 4 qua ví dụ đề


**Độ phức tạp.** Mỗi phần tử một truy vấn + một cập nhật, mỗi thao tác O(log m): **Time O(n log n)** (cộng sắp xếp để nén tọa độ O(n log n)). **Space O(n)**.

### 7.4. Cách 5 — Xếp tầng (patience) + tổng tiền tố đếm, O(n log n)

**Ý tưởng.** Kỹ thuật LIS O(n log n) quen thuộc giữ `tails[k]` = giá trị cuối nhỏ nhất của dãy tăng độ dài `k+1`. Để **đếm**, ta mở rộng: với mỗi độ dài ta giữ **cả danh sách** các giá trị cuối đã gặp (không chỉ giá trị nhỏ nhất), cùng số cách kết thúc ở mỗi giá trị đó.

**Hai tính chất then chốt:**

1. Trong mỗi tầng `d[k]` (dãy độ dài `k+1`), các giá trị cuối được thêm vào theo thứ tự **giảm dần** (không tăng). Vì phần tử mới `v` được xếp vào tầng đầu tiên mà giá trị cuối `<` ... (tầng `k` đầu tiên có `d[k][-1] >= v`), nên nó **không lớn hơn** phần tử cuối cũ của tầng đó.
2. Số cách kết thúc ở `v` với độ dài `k+1` = tổng số cách của các phần tử ở tầng **`k-1`** (độ dài `k`) có giá trị **`< v`**. Vì tầng `k-1` giảm dần, các phần tử `< v` tạo thành một **hậu tố** của list; ta tìm ranh giới bằng nhị phân và lấy tổng nhờ **mảng tổng tiền tố** `cnt[k-1]`.

**Giải thích từng đoạn.**

1. Với `v`, tìm nhị phân tầng đầu tiên `k` mà `d[k][-1] >= v` (nếu không có, `k = len(d)` → tạo tầng mới).
2. Nếu `k == 0`: số cách `c = 1`. Ngược lại tìm trong `d[k-1]` (giảm dần) vị trí `l` đầu tiên mà `d[k-1][l] < v`, và `c = cnt[k-1][-1] - cnt[k-1][l]` (mảng tiền tố có thêm số `0` ở đầu).
3. Thêm `v` vào `d[k]` và thêm `cnt[k][-1] + c` vào tiền tố.
4. Đáp án là tổng số cách ở tầng cuối cùng `cnt[-1][-1]`.

In [17]:
def patience_count(nums):
    d = []        # d[k]: các giá trị cuối của dãy độ dài k+1, theo thứ tự GIẢM dần
    cnt = []      # cnt[k]: mảng tổng tiền tố số cách, bắt đầu bằng 0
    for v in nums:
        lo, hi = 0, len(d)                 # tầng đầu tiên có d[k][-1] >= v
        while lo < hi:
            mid = (lo + hi) // 2
            if d[mid][-1] >= v:
                hi = mid
            else:
                lo = mid + 1
        k = lo
        c = 1
        if k > 0:
            arr = d[k - 1]                 # giảm dần: tìm vị trí đầu tiên có giá trị < v
            l, r = 0, len(arr)
            while l < r:
                mid = (l + r) // 2
                if arr[mid] < v:
                    r = mid
                else:
                    l = mid + 1
            c = cnt[k - 1][-1] - cnt[k - 1][l]
        if k == len(d):
            d.append([v])
            cnt.append([0, c])
        else:
            d[k].append(v)
            cnt[k].append(cnt[k][-1] + c)
    return cnt[-1][-1]

for nums, expected in examples:
    assert patience_count(nums) == expected
    print(nums, "->", patience_count(nums))

[1, 3, 5, 4, 7] -> 2
[2, 2, 2, 2, 2] -> 5


In [18]:
# Code nộp LeetCode — Cách 5: xếp tầng + tổng tiền tố
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        d = []
        cnt = []
        for v in nums:
            lo, hi = 0, len(d)
            while lo < hi:
                mid = (lo + hi) // 2
                if d[mid][-1] >= v:
                    hi = mid
                else:
                    lo = mid + 1
            k = lo
            c = 1
            if k > 0:
                arr = d[k - 1]
                l, r = 0, len(arr)
                while l < r:
                    mid = (l + r) // 2
                    if arr[mid] < v:
                        r = mid
                    else:
                        l = mid + 1
                c = cnt[k - 1][-1] - cnt[k - 1][l]
            if k == len(d):
                d.append([v])
                cnt.append([0, c])
            else:
                d[k].append(v)
                cnt[k].append(cnt[k][-1] + c)
        return cnt[-1][-1]

for nums, expected in examples:
    assert Solution().findNumberOfLIS(nums) == expected
print("Cách 5 qua ví dụ đề")

Cách 5 qua ví dụ đề


**Độ phức tạp.** Mỗi phần tử hai lần nhị phân: **Time O(n log n)**. Các list `d`, `cnt` tổng cộng `O(n)` phần tử: **Space O(n)**. Cách này tinh tế hơn BIT nhưng khó tự chứng minh; nên nắm BIT trước.

## 8. TRACE từng bước

**Trace cách 2 (DP)** với `nums = [1,3,5,4,7]`. Mỗi dòng là một cặp `(j, i)` hợp lệ (`nums[j] < nums[i]`):

| i | nums[i] | j | nums[j] | f[j]+1 | so với f[i] | hành động | f[i] | cnt[i] |
|---|---|---|---|---|---|---|---|---|
| 0 | 1 | — | | | | khởi tạo | 1 | 1 |
| 1 | 3 | 0 | 1 | 2 | 1 < 2 | đổi: f=2, cnt=cnt[0]=1 | 2 | 1 |
| 2 | 5 | 0 | 1 | 2 | 1 < 2 | đổi: f=2, cnt=1 | 2 | 1 |
| 2 | 5 | 1 | 3 | 3 | 2 < 3 | đổi: f=3, cnt=cnt[1]=1 | 3 | 1 |
| 3 | 4 | 0 | 1 | 2 | 1 < 2 | đổi: f=2, cnt=1 | 2 | 1 |
| 3 | 4 | 1 | 3 | 3 | 2 < 3 | đổi: f=3, cnt=1 | 3 | 1 |
| 3 | 4 | 2 | 5 | — | 5 < 4 sai | bỏ qua | 3 | 1 |
| 4 | 7 | 0 | 1 | 2 | 1 < 2 | đổi: f=2, cnt=1 | 2 | 1 |
| 4 | 7 | 1 | 3 | 3 | 2 < 3 | đổi: f=3, cnt=1 | 3 | 1 |
| 4 | 7 | 2 | 5 | 4 | 3 < 4 | đổi: f=4, cnt=cnt[2]=1 | 4 | 1 |
| 4 | 7 | 3 | 4 | 4 | 4 == 4 | cộng: cnt += cnt[3] | 4 | **2** |

Kết quả: `f = [1,2,3,3,4]`, `cnt = [1,1,1,1,2]`, `mx = 4`, đáp án `2`.

In [19]:
def trace_dp(nums):
    n = len(nums)
    f = [1] * n
    cnt = [1] * n
    print("nums =", nums)
    for i in range(n):
        print(f"--- i={i} (nums[i]={nums[i]}) khởi tạo f=1 cnt=1")
        for j in range(i):
            if nums[j] < nums[i]:
                cand = f[j] + 1
                if f[i] < cand:
                    f[i] = cand
                    cnt[i] = cnt[j]
                    act = f"độ dài tốt hơn -> f={f[i]}, cnt={cnt[i]} (đặt lại)"
                elif f[i] == cand:
                    cnt[i] += cnt[j]
                    act = f"cùng độ dài -> cnt += cnt[{j}]={cnt[j]} => {cnt[i]}"
                else:
                    act = "ngắn hơn -> bỏ qua"
                print(f"  j={j} nums[j]={nums[j]} < {nums[i]}: ứng viên {cand} | {act}")
            else:
                print(f"  j={j} nums[j]={nums[j]} >= {nums[i]}: không nối được")
    mx = max(f)
    ans = sum(c for fi, c in zip(f, cnt) if fi == mx)
    print("f   =", f)
    print("cnt =", cnt)
    print(f"mx = {mx} -> đáp án = {ans}")
    return ans

assert trace_dp([1, 3, 5, 4, 7]) == 2

nums = [1, 3, 5, 4, 7]
--- i=0 (nums[i]=1) khởi tạo f=1 cnt=1
--- i=1 (nums[i]=3) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 3: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
--- i=2 (nums[i]=5) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 5: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 < 5: ứng viên 3 | độ dài tốt hơn -> f=3, cnt=1 (đặt lại)
--- i=3 (nums[i]=4) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 4: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 < 4: ứng viên 3 | độ dài tốt hơn -> f=3, cnt=1 (đặt lại)
  j=2 nums[j]=5 >= 4: không nối được
--- i=4 (nums[i]=7) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 7: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 < 7: ứng viên 3 | độ dài tốt hơn -> f=3, cnt=1 (đặt lại)
  j=2 nums[j]=5 < 7: ứng viên 4 | độ dài tốt hơn -> f=4, cnt=1 (đặt lại)
  j=3 nums[j]=4 < 7: ứng viên 4 | cùng độ dài -> cnt += cnt[3]=1 => 2
f   = [1, 2, 3, 3, 4]
cnt = [1, 1, 1, 1, 2]
mx = 4 -> đáp án = 2


Trace tiếp ví dụ **có số trùng và nhiều nhánh**: `nums = [1,3,2,4,3,5]`. Ta kỳ vọng `f = [1,2,2,3,3,4]`, `cnt = [1,1,1,2,1,3]`, đáp án `3`: các dãy `[1,3,4,5]`, `[1,2,4,5]`, `[1,2,3,5]`. Chú ý `i = 5` (số `5`): từ `j=3` (`4`, `f=3`, `cnt=2`) cho cnt = 2, từ `j=4` (`3`, `f=3`, `cnt=1`) cộng thêm 1 → `3`.

In [20]:
assert trace_dp([1, 3, 2, 4, 3, 5]) == 3

nums = [1, 3, 2, 4, 3, 5]
--- i=0 (nums[i]=1) khởi tạo f=1 cnt=1
--- i=1 (nums[i]=3) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 3: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
--- i=2 (nums[i]=2) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 2: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 >= 2: không nối được
--- i=3 (nums[i]=4) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 4: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 < 4: ứng viên 3 | độ dài tốt hơn -> f=3, cnt=1 (đặt lại)
  j=2 nums[j]=2 < 4: ứng viên 3 | cùng độ dài -> cnt += cnt[2]=1 => 2
--- i=4 (nums[i]=3) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 3: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 >= 3: không nối được
  j=2 nums[j]=2 < 3: ứng viên 3 | độ dài tốt hơn -> f=3, cnt=1 (đặt lại)
  j=3 nums[j]=4 >= 3: không nối được
--- i=5 (nums[i]=5) khởi tạo f=1 cnt=1
  j=0 nums[j]=1 < 5: ứng viên 2 | độ dài tốt hơn -> f=2, cnt=1 (đặt lại)
  j=1 nums[j]=3 < 5: ứng viên 3 | độ dài tốt hơn

**Trace cách 5 (xếp tầng)** với `nums = [1,3,5,4,7]`:

| v | tầng k (chọn) | tầng trước k-1 (giảm dần) | phần tử `< v` | c | `d` sau | `cnt` sau |
|---|---|---|---|---|---|---|
| 1 | 0 (mới) | — | — | 1 | [[1]] | [[0,1]] |
| 3 | 1 (mới) | [1] | [1] | 1 | [[1],[3]] | [[0,1],[0,1]] |
| 5 | 2 (mới) | [3] | [3] | 1 | [[1],[3],[5]] | [[0,1],[0,1],[0,1]] |
| 4 | 2 (`5 >= 4`) | [3] | [3] | 1 | tầng 2 = [5,4] | tầng 2 = [0,1,2] |
| 7 | 3 (mới) | [5,4] | [5,4] cả hai | 2 | [..., [7]] | [..., [0,2]] |

Đáp án `cnt[-1][-1] = 2`. Cho máy in lại từng bước.

In [21]:
def trace_patience(nums):
    d, cnt = [], []
    print("nums =", nums)
    for v in nums:
        lo, hi = 0, len(d)
        while lo < hi:
            mid = (lo + hi) // 2
            if d[mid][-1] >= v:
                hi = mid
            else:
                lo = mid + 1
        k = lo
        c = 1
        if k > 0:
            arr = d[k - 1]
            l, r = 0, len(arr)
            while l < r:
                mid = (l + r) // 2
                if arr[mid] < v:
                    r = mid
                else:
                    l = mid + 1
            c = cnt[k - 1][-1] - cnt[k - 1][l]
            lower = arr[l:]
        else:
            lower = []
        if k == len(d):
            d.append([v]); cnt.append([0, c])
        else:
            d[k].append(v); cnt[k].append(cnt[k][-1] + c)
        print(f"v={v}: tầng={k} | phần tử < v ở tầng trước: {lower} | c={c} | d={d} | cnt={cnt}")
    print("đáp án:", cnt[-1][-1])
    return cnt[-1][-1]

assert trace_patience([1, 3, 5, 4, 7]) == 2

nums = [1, 3, 5, 4, 7]
v=1: tầng=0 | phần tử < v ở tầng trước: [] | c=1 | d=[[1]] | cnt=[[0, 1]]
v=3: tầng=1 | phần tử < v ở tầng trước: [1] | c=1 | d=[[1], [3]] | cnt=[[0, 1], [0, 1]]
v=5: tầng=2 | phần tử < v ở tầng trước: [3] | c=1 | d=[[1], [3], [5]] | cnt=[[0, 1], [0, 1], [0, 1]]
v=4: tầng=2 | phần tử < v ở tầng trước: [3] | c=1 | d=[[1], [3], [5, 4]] | cnt=[[0, 1], [0, 1], [0, 1, 2]]
v=7: tầng=3 | phần tử < v ở tầng trước: [5, 4] | c=2 | d=[[1], [3], [5, 4], [7]] | cnt=[[0, 1], [0, 1], [0, 1, 2], [0, 2]]
đáp án: 2


## 9. Code nộp LeetCode

Ta nộp **cách 2 (DP O(n²))**: ngắn, dễ kiểm chứng và đủ nhanh với `n ≤ 2000`. Nếu muốn O(n log n) thì dùng cách 4 (BIT).

In [22]:
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        n = len(nums)
        f = [1] * n
        cnt = [1] * n
        mx = ans = 0
        for i in range(n):
            for j in range(i):
                if nums[j] < nums[i]:
                    if f[i] < f[j] + 1:
                        f[i] = f[j] + 1
                        cnt[i] = cnt[j]
                    elif f[i] == f[j] + 1:
                        cnt[i] += cnt[j]
            if f[i] > mx:
                mx, ans = f[i], cnt[i]
            elif f[i] == mx:
                ans += cnt[i]
        return ans


sol = Solution()
for nums, expected in examples:
    got = sol.findNumberOfLIS(nums)
    print(nums, "->", got, "(kỳ vọng", expected, ")")
    assert got == expected

[1, 3, 5, 4, 7] -> 2 (kỳ vọng 2 )
[2, 2, 2, 2, 2] -> 5 (kỳ vọng 5 )


## 10. Vì sao đúng

**Định nghĩa.** Với mỗi `i`, tập `S_i` là tập mọi dãy con tăng ngặt kết thúc **đúng tại chỉ số `i`**. `f[i]` là độ dài lớn nhất trong `S_i`, `cnt[i]` là số dãy đạt độ dài đó.

**Công thức truy hồi.** Một dãy trong `S_i` độ dài `≥ 2` có phần tử áp chót tại chỉ số `j < i` với `nums[j] < nums[i]`, và phần còn lại (bỏ `nums[i]`) là một dãy thuộc `S_j`. Ngược lại, nối `nums[i]` vào bất kỳ dãy thuộc `S_j` (`nums[j] < nums[i]`) ta được một dãy thuộc `S_i`. Vì vậy:

- Độ dài lớn nhất trong `S_i` là `1 + max f[j]` (hoặc 1 nếu không có `j`).
- Các dãy đạt độ dài lớn nhất trong `S_i` ứng với các cặp `(j, dãy dài nhất trong S_j)` mà `f[j]` đạt `max f[j]`. Hai cặp khác `j` cho dãy khác (khác phần tử áp chót) và cùng `j` cho dãy khác khi phần còn lại khác → **không đếm trùng**. Do đó `cnt[i] = Σ cnt[j]` trên các `j` đạt max, đúng với cách "đặt lại khi tốt hơn, cộng dồn khi bằng nhau".

**Một lưu ý quan trọng về "dãy dài nhất kết thúc tại `j`".** Khi nối `nums[i]` vào một dãy trong `S_j` **không** dài nhất của `S_j` thì độ dài `< f[j] + 1`, không thể đạt max của `S_i` (vì đã có ứng viên `f[j]+1` từ cùng `j`). Nên chỉ cần quan tâm đến các dãy dài nhất của mỗi `S_j`.

**Đáp án.** Mọi dãy tăng dài nhất toàn cục (độ dài `mx = max f`) kết thúc tại đúng một chỉ số `i` với `f[i] = mx`, nên số dãy là `Σ cnt[i]` trên các `i` có `f[i] = mx`. Tiến trình giữ `mx`, `ans` chạy cùng lúc cho kết quả này.

**Ngặt:** điều kiện `nums[j] < nums[i]` (không `<=`) đảm bảo "tăng ngặt". Với mảng `[2,2,2,2,2]` không có cặp nào hợp lệ nên `f = [1]*5`, `cnt = [1]*5`, đáp án `5` — đúng với đề.

**Cách 4 và 5** tính cùng cặp `(f, cnt)`, chỉ khác cách lấy "gộp mọi `j < i`, `nums[j] < nums[i]`": BIT gộp theo tiền tố giá trị; xếp tầng dùng cấu trúc đơn điệu của các tầng.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Liệt kê dãy con | O(2^n) | O(n) | chỉ n ≤ ~25 |
| 2. DP hai mảng | O(n²) | O(n) | `n = 2000`: ~2 triệu bước |
| 3. DFS có nhớ | O(n²) | O(n) | cần nâng giới hạn đệ quy |
| 4. Fenwick tree | O(n log n) | O(n) | cần nén tọa độ |
| 5. Xếp tầng + tiền tố | O(n log n) | O(n) | tinh tế, khó chứng minh |

Với giới hạn `n ≤ 2000`, cách 2 là đủ; cách 4/5 dùng khi `n` lên tới ~10^5.

## 12. Bộ test

Ta so **cả 5 cách** với `oracle` vét cạn trên ca cố định và ngẫu nhiên, rồi so các cách nhanh với nhau trên `n` lớn.

In [23]:
all_ways = {
    "brute_force": brute_force,
    "dp_quadratic": dp_quadratic,
    "memo_dfs": memo_dfs,
    "bit_way": bit_way,
    "patience_count": patience_count,
}

tests = [
    ([1, 3, 5, 4, 7], 2),
    ([2, 2, 2, 2, 2], 5),
    ([1], 1),
    ([1, 2], 1),
    ([2, 1], 2),                       # độ dài 1, hai dãy
    ([1, 2, 3, 4, 5], 1),              # tăng hẳn: duy nhất cả mảng
    ([5, 4, 3, 2, 1], 5),              # giảm hẳn: mỗi phần tử một dãy
    ([1, 3, 2, 4, 3, 5], 3),
    ([1, 2, 4, 3, 5, 4, 7, 2], 3),
    ([3, 1, 2], 1),
    ([2, 1, 3, 1, 4], 2),
    ([-3, -1, -2, 0], 2),              # số âm
    ([1, 1, 2, 2], 4),                 # trùng giá trị: 2 cách chọn "1" x 2 cách chọn "2"
    ([1, 2, 1, 2], 3),
]
for nums, expected in tests:
    assert oracle(nums) == expected, (nums, oracle(nums), expected)
    for name, fn in all_ways.items():
        assert fn(list(nums)) == expected, (name, nums, fn(list(nums)))
print("OK:", len(tests), "ca cố định, 5 cách khớp oracle")

OK: 14 ca cố định, 5 cách khớp oracle


### Test ngẫu nhiên nhỏ (so với oracle)

Dùng seed cố định, giá trị nhỏ `-3..3` để **nhiều số trùng**, độ dài `1..10` (oracle O(2^n) nên giữ n nhỏ).

In [24]:
random.seed(673)
cnt_tests = 0
for _ in range(1500):
    n = random.randint(1, 10)
    nums = [random.randint(-3, 3) for _ in range(n)]
    want = oracle(nums)
    for name, fn in all_ways.items():
        got = fn(list(nums))
        assert got == want, (name, nums, got, want)
    cnt_tests += 1
print("OK:", cnt_tests, "mảng ngẫu nhiên nhỏ")

OK: 1500 mảng ngẫu nhiên nhỏ


### Test ngẫu nhiên vừa (so các cách nhanh với nhau)

`n` tới 60, giá trị trong khoảng rộng `[-10^6, 10^6]` hoặc hẹp (nhiều trùng). Cách 1 và `oracle` quá chậm nên so cách 2 với 3, 4, 5.

In [25]:
random.seed(6730)
for _ in range(400):
    n = random.randint(1, 60)
    if random.random() < 0.5:
        nums = [random.randint(-10**6, 10**6) for _ in range(n)]
    else:
        nums = [random.randint(0, 6) for _ in range(n)]
    ref = dp_quadratic(nums)
    assert memo_dfs(nums) == ref
    assert bit_way(nums) == ref
    assert patience_count(nums) == ref
print("OK: 400 mảng cỡ vừa")

OK: 400 mảng cỡ vừa


### Test tối đa `n = 2000`

Bốn ca: ngẫu nhiên rộng, ngẫu nhiên hẹp (nhiều trùng, đáp án có thể rất lớn), tăng hẳn, giảm hẳn, toàn bằng nhau. Chỉ cần so các cách O(n²) với O(n log n). (Đề đảm bảo đáp án vừa 32-bit; với test hẹp ngẫu nhiên Python vẫn tính chính xác vì số nguyên lớn tùy ý.)

In [26]:
random.seed(7)
n = 2000
cases = {
    "ngẫu nhiên rộng": [random.randint(-10**6, 10**6) for _ in range(n)],
    "ngẫu nhiên hẹp (0..30)": [random.randint(0, 30) for _ in range(n)],
    "tăng hẳn": list(range(n)),
    "giảm hẳn": list(range(n, 0, -1)),
    "toàn bằng nhau": [5] * n,
}
expected_known = {"tăng hẳn": 1, "giảm hẳn": n, "toàn bằng nhau": n}
for name, nums in cases.items():
    ref = dp_quadratic(nums)
    assert memo_dfs(nums) == ref, name
    assert bit_way(nums) == ref, name
    assert patience_count(nums) == ref, name
    if name in expected_known:
        assert ref == expected_known[name], name
    print(f"{name}: đáp án = {ref}")
print("OK: n = 2000, các cách khớp nhau")

ngẫu nhiên rộng: đáp án = 847872


ngẫu nhiên hẹp (0..30): đáp án = 9764101259341137527948


tăng hẳn: đáp án = 1
giảm hẳn: đáp án = 2000


toàn bằng nhau: đáp án = 2000
OK: n = 2000, các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<=` thay `<` (không còn "tăng ngặt")

Đề yêu cầu **tăng ngặt**. Nếu cho phép bằng nhau (`nums[j] <= nums[i]`), ta đếm dãy **không giảm**: `[2,2,2,2,2]` ra `1` (cả mảng) thay vì `5`.

In [27]:
def wrong_non_strict(nums):
    n = len(nums)
    f, cnt = [1] * n, [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] <= nums[i]:            # SAI: phải là <
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

print("sai    :", wrong_non_strict([2, 2, 2, 2, 2]), "| đúng:", dp_quadratic([2, 2, 2, 2, 2]))
assert wrong_non_strict([2, 2, 2, 2, 2]) == 1
assert dp_quadratic([2, 2, 2, 2, 2]) == 5

sai    : 1 | đúng: 5


**Sửa:** điều kiện `nums[j] < nums[i]`.

### Lỗi 2 — Quên **đặt lại** `cnt[i]` khi tìm được độ dài tốt hơn

Khi `f[i] < f[j] + 1`, những dãy đếm trước đó có độ dài **ngắn hơn** nên phải bị bỏ: `cnt[i] = cnt[j]`. Nếu chỉ cộng `cnt[i] += cnt[j]` thì cộng lẫn cả các dãy ngắn.

In [28]:
def wrong_no_reset(nums):
    n = len(nums)
    f, cnt = [1] * n, [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i] = f[j] + 1
                    cnt[i] += cnt[j]          # SAI: phải là cnt[i] = cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

print("sai    :", wrong_no_reset([1, 3, 5, 4, 7]), "| đúng:", dp_quadratic([1, 3, 5, 4, 7]))
assert wrong_no_reset([1, 3, 5, 4, 7]) != 2

sai    : 12 | đúng: 2


**Vì sao sai?** Ban đầu `cnt[i] = 1` đại diện cho dãy chỉ gồm `nums[i]` (độ dài 1). Khi có độ dài tốt hơn, dãy ngắn đó không còn dài nhất, phải **thay** chứ không cộng.

### Lỗi 3 — Khởi tạo `cnt = 0`

`cnt[i]` phải bắt đầu bằng `1` (dãy một phần tử). Nếu bằng `0` thì số cách nhân lên từ `0` luôn ra `0` — ví dụ `[2,2,2,2,2]` ra `0`.

In [29]:
def wrong_cnt_zero(nums):
    n = len(nums)
    f, cnt = [1] * n, [0] * n                  # SAI: cnt khởi tạo 0
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

print("sai    :", wrong_cnt_zero([2, 2, 2, 2, 2]), "| đúng:", dp_quadratic([2, 2, 2, 2, 2]))
assert wrong_cnt_zero([2, 2, 2, 2, 2]) == 0
assert wrong_cnt_zero([1, 3, 5, 4, 7]) == 0

sai    : 0 | đúng: 5


### Lỗi 4 — Trả `cnt[-1]` hoặc `cnt[argmax]` thay vì **cộng** mọi vị trí đạt độ dài lớn nhất

Các dãy dài nhất có thể kết thúc ở **nhiều** vị trí khác nhau. Ví dụ `[1,2,1,2]` có 3 dãy dài nhất `[1,2]` (các cặp chỉ số `(0,1)`, `(0,3)`, `(2,3)`) kết thúc ở chỉ số 1 **và** 3.

In [30]:
def wrong_last_only(nums):
    n = len(nums)
    f, cnt = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
    return cnt[-1]                              # SAI: chỉ lấy vị trí cuối

nums = [1, 2, 1, 2]
print("sai    :", wrong_last_only(nums), "| đúng:", dp_quadratic(nums), "| oracle:", oracle(nums))
assert wrong_last_only(nums) != dp_quadratic(nums)

sai    : 2 | đúng: 3 | oracle: 3


**Vì sao sai?** `[1,2,1,2]`: dãy dài nhất có độ dài 2: `[1,2]` với các cặp chỉ số `(0,1)`, `(0,3)`, `(2,3)` — kết thúc ở chỉ số 1 **và** 3. Cần **cộng `cnt[i]` của mọi `i` có `f[i] = mx`** (hoặc giữ `mx`, `ans` chạy như code đúng).

### Lỗi 5 — Nhầm với bài tìm **độ dài** LIS

Bài 300 hỏi độ dài LIS (`max(f)`); bài này hỏi **số lượng** dãy. Trả `max(f)` cho ví dụ 1 ra `4` thay vì `2`. Hãy đọc kỹ câu hỏi: "return the **number of** longest increasing subsequences".

In [31]:
def wrong_return_length(nums):
    n = len(nums)
    f = [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                f[i] = max(f[i], f[j] + 1)
    return max(f)                               # SAI: đây là độ dài, không phải số dãy

print("sai    :", wrong_return_length([1, 3, 5, 4, 7]), "| đúng:", dp_quadratic([1, 3, 5, 4, 7]))
assert wrong_return_length([1, 3, 5, 4, 7]) == 4

sai    : 4 | đúng: 2


## 14. Mẹo & nhận dạng pattern

- **"Đếm số cách đạt giá trị tối ưu"** → DP hai mảng: một mảng cho **giá trị tối ưu**, một mảng cho **số cách đạt tối ưu**. Quy tắc cập nhật luôn là: **tốt hơn → thay**, **bằng → cộng**, **kém hơn → bỏ**. Mẫu này xuất hiện ở "số đường đi ngắn nhất" (Dijkstra/BFS đếm đường), "số cách đổi tiền tối thiểu", v.v.
- **Dãy con tăng + "kết thúc tại i"** là trạng thái chuẩn của họ LIS (bài 300, 354, 646, 673).
- **O(n²) đủ khi `n ≤ 2000`**; nếu `n ~ 10^5` thì nghĩ BIT / segment tree trên **giá trị** (đã nén tọa độ).
- **Số trùng**: chú ý "ngặt" hay "không giảm" — chỉ đổi `<` thành `<=` ở truy vấn (và BIT thì query tới `idx[x]` thay vì `idx[x]-1`).
- Với "hai dãy khác nhau khi khác **chỉ số**" thì các phần tử trùng giá trị vẫn tạo ra dãy khác nhau — đừng khử trùng mảng!
- Luôn viết `oracle` 2^n cho `n ≤ 10` và so với lời giải trên mảng ngẫu nhiên có **nhiều số trùng**: bắt lỗi `<`/`<=` và quên đặt lại rất hiệu quả.
- Biến `mx`, `ans` chạy cùng lúc trong vòng `for i` tránh phải quét lại `f` lần cuối.

## 15. Biến thể

### 15.1. Chỉ cần **độ dài** LIS (bài 300) — O(n log n) với `bisect_left`

Giữ `tails[k]` là giá trị cuối nhỏ nhất của dãy tăng ngặt độ dài `k+1`; với mỗi `x` dùng `bisect_left` để thay hoặc nối thêm.

In [32]:
def lis_length(nums):
    tails = []
    for x in nums:
        k = bisect_left(tails, x)          # bisect_left -> tăng NGẶT
        if k == len(tails):
            tails.append(x)
        else:
            tails[k] = x
    return len(tails)

print(lis_length([1, 3, 5, 4, 7]), lis_length([2, 2, 2, 2, 2]))
assert lis_length([1, 3, 5, 4, 7]) == 4
assert lis_length([2, 2, 2, 2, 2]) == 1
random.seed(15)
for _ in range(200):
    a = [random.randint(-3, 3) for _ in range(random.randint(1, 9))]
    n = len(a)
    ref = max(bin(m).count("1") for m in range(1, 1 << n)
              if is_strictly_increasing([a[i] for i in range(n) if m >> i & 1]))
    assert lis_length(a) == ref
print("lis_length khớp vét cạn")

4 1
lis_length khớp vét cạn


### 15.2. Đếm dãy con **không giảm** dài nhất (cho phép bằng nhau)

Chỉ cần đổi điều kiện nối thành `nums[j] <= nums[i]`.

In [33]:
def count_non_decreasing(nums):
    n = len(nums)
    f, cnt = [1] * n, [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] <= nums[i]:
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

print(count_non_decreasing([2, 2, 2, 2, 2]))        # 1: cả mảng
print(count_non_decreasing([1, 3, 5, 4, 7]))        # 2
assert count_non_decreasing([2, 2, 2, 2, 2]) == 1
assert count_non_decreasing([1, 2, 1, 2]) == 2      # độ dài 3: [1,1,2] (0,2,3) và [1,2,2] (0,1,3)

1
2


### 15.3. Dựng lại **một** dãy tăng dài nhất

Thêm mảng `prev[i]` ghi `j` đã cho `f[i]` tốt nhất, rồi lần ngược từ `i` có `f[i]` lớn nhất.

In [34]:
def one_lis(nums):
    n = len(nums)
    f = [1] * n
    prev = [-1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i] and f[j] + 1 > f[i]:
                f[i] = f[j] + 1
                prev[i] = j
    i = max(range(n), key=lambda k: f[k])
    seq = []
    while i != -1:
        seq.append(nums[i])
        i = prev[i]
    return seq[::-1]

res = one_lis([1, 3, 5, 4, 7])
print(res)
assert len(res) == 4 and is_strictly_increasing(res)

[1, 3, 5, 7]


### 15.4. Bài liên quan

- **300. Longest Increasing Subsequence** — chỉ độ dài (mục 15.1).
- **334. Increasing Triplet Subsequence** — tồn tại dãy tăng độ dài 3.
- **354. Russian Doll Envelopes** — LIS hai chiều.
- **646. Maximum Length of Pair Chain** — họ hàng của LIS.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê dãy con | thử chọn/bỏ từng phần tử | O(2^n) | O(n) | chỉ để hiểu đề và làm oracle |
| 2. DP `f`, `cnt` | `(độ dài, số cách)` kết thúc tại i, nhìn lui mọi j | O(n²) | O(n) | **nên nộp** khi n ≤ 2000 |
| 3. DFS có nhớ | `(độ dài, số cách)` bắt đầu tại i | O(n²) | O(n) | thích đệ quy, nhớ nâng giới hạn |
| 4. Fenwick tree | truy vấn tiền tố trên giá trị với phép gộp `(L, C)` | O(n log n) | O(n) | n lớn, cần tối ưu, biết BIT |
| 5. Xếp tầng + tiền tố | các tầng giảm dần, đếm bằng tổng hậu tố | O(n log n) | O(n) | thi đấu, thích gọn nhưng cần hiểu kỹ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` — một tuple `(f, cnt)` — rồi điền vào `my_guess_1`.

In [35]:
nums = [3, 1, 2]
n = len(nums)
f, cnt = [1] * n, [1] * n
for i in range(n):
    for j in range(i):
        if nums[j] < nums[i]:
            if f[i] < f[j] + 1:
                f[i], cnt[i] = f[j] + 1, cnt[j]
            elif f[i] == f[j] + 1:
                cnt[i] += cnt[j]
result = (f, cnt)
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền tuple dạng ([...], [...])

### Bài 2 (tính tay)

Với `nums = [1, 2, 4, 3, 5, 4, 7, 2]`, hãy tính tay mảng `f` và `cnt` (có thể vẽ bảng như mục 8) rồi điền độ dài LIS vào `my_len` và số dãy vào `my_count`.

In [36]:
my_len = None       # TODO: độ dài dãy tăng dài nhất
my_count = None     # TODO: số dãy tăng dài nhất

### Bài 3 (điền TODO)

Hoàn thành phần lõi của DP: thay các `...` bằng điều kiện / phép gán đúng.

In [37]:
def exercise3(nums):
    n = len(nums)
    f = [1] * n
    cnt = [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if ...:                       # TODO: điều kiện nối được (tăng NGẶT)
                if f[i] < f[j] + 1:
                    ...                   # TODO: cập nhật f[i] và cnt[i] (nhớ đặt lại)
                elif f[i] == f[j] + 1:
                    ...                   # TODO: cộng dồn số cách
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

### Bài 4 (tự viết lại)

Tự viết `exercise4(nums)` giải bài bằng **DFS có nhớ theo hướng "bắt đầu tại i"** (cách 3) — không nhìn lại code. Gợi ý: hàm `dfs(i)` trả `(độ dài, số cách)`.

In [38]:
def exercise4(nums):
    # TODO: viết lời giải
    pass

### Bài 5 (biến thể / mini challenge)

Viết `exercise5(nums)` trả về tuple `(L, C)`: **độ dài** dãy tăng ngặt dài nhất và **số lượng** dãy đó. Ví dụ `[1,3,5,4,7]` → `(4, 2)`, `[2,2,2,2,2]` → `(1, 5)`.

In [39]:
def exercise5(nums):
    # TODO: viết lời giải, trả về (L, C)
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: duyệt `i` từ 0, với mỗi `i` xét mọi `j < i`.
- Mức 2: chỉ khi `nums[j] < nums[i]` mới nối được. `nums = [3,1,2]`: cặp hợp lệ duy nhất là `(j=1, i=2)` vì `1 < 2`.
- Mức 3: `i = 2` có `f = f[1] + 1 = 2`, `cnt = cnt[1] = 1`; hai phần tử còn lại giữ `(1, 1)`.

**Bài 2**
- Mức 1: vẽ bảng như mục 8, từng `i` một.
- Mức 2: LIS dài ra từ `1 → 2 → ...`; tìm các điểm có hai nguồn cùng độ dài (ví dụ `5` nhận từ `4` hoặc `3`).
- Mức 3: kiểm tra chéo bằng cách liệt kê các dãy dài nhất: chúng đều chứa `1, 2` ở đầu và kết thúc bằng `7`; rồi so với `oracle`.

**Bài 3**
- Mức 1: nhìn lại mục 7.1.
- Mức 2: điều kiện là `nums[j] < nums[i]`; nhánh "tốt hơn" phải **thay** `cnt[i]`, nhánh "bằng" mới **cộng**.
- Mức 3: `f[i], cnt[i] = f[j] + 1, cnt[j]` và `cnt[i] += cnt[j]`.

**Bài 4**
- Mức 1: định nghĩa `dfs(i)` = cặp tốt nhất của dãy **bắt đầu** tại `i`; khởi tạo `(1, 1)`.
- Mức 2: với `j > i`, `nums[j] > nums[i]`, ứng viên `(L_j + 1, C_j)`; gộp: độ dài lớn hơn thắng, bằng nhau thì cộng.
- Mức 3: nhớ `memo`, nâng `sys.setrecursionlimit`, và gộp `dfs(i)` qua mọi `i` để ra đáp án.

**Bài 5**
- Mức 1: giống code nộp, nhưng cuối trả cả `mx` lẫn `ans`.
- Mức 2: giữ `mx`, `ans` chạy cùng vòng `for i`.
- Mức 3: `return mx, ans`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`.

In [40]:
# Bài 1
nums = [3, 1, 2]
n = len(nums)
f, cnt = [1] * n, [1] * n
for i in range(n):
    for j in range(i):
        if nums[j] < nums[i]:
            if f[i] < f[j] + 1:
                f[i], cnt[i] = f[j] + 1, cnt[j]
            elif f[i] == f[j] + 1:
                cnt[i] += cnt[j]
answer_1 = (f, cnt)
print("Bài 1:", answer_1)
assert answer_1 == ([1, 1, 2], [1, 1, 1])

# Bài 2
nums2 = [1, 2, 4, 3, 5, 4, 7, 2]
n = len(nums2)
f2, c2 = [1] * n, [1] * n
for i in range(n):
    for j in range(i):
        if nums2[j] < nums2[i]:
            if f2[i] < f2[j] + 1:
                f2[i], c2[i] = f2[j] + 1, c2[j]
            elif f2[i] == f2[j] + 1:
                c2[i] += c2[j]
print("Bài 2: f =", f2, ", cnt =", c2)
assert f2 == [1, 2, 3, 3, 4, 4, 5, 2]
assert c2 == [1, 1, 1, 1, 2, 1, 3, 1]
assert (max(f2), 3) == (5, 3) and oracle(nums2) == 3

Bài 1: ([1, 1, 2], [1, 1, 1])
Bài 2: f = [1, 2, 3, 3, 4, 4, 5, 2] , cnt = [1, 1, 1, 1, 2, 1, 3, 1]


In [41]:
# Bài 3
def exercise3_solution(nums):
    n = len(nums)
    f = [1] * n
    cnt = [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return ans

# Bài 4
def exercise4_solution(nums):
    n = len(nums)
    old = sys.getrecursionlimit()
    sys.setrecursionlimit(max(old, n + 100))
    memo = {}

    def dfs(i):
        if i in memo:
            return memo[i]
        best = (1, 1)
        for j in range(i + 1, n):
            if nums[j] > nums[i]:
                L, C = dfs(j)
                if L + 1 > best[0]:
                    best = (L + 1, C)
                elif L + 1 == best[0]:
                    best = (best[0], best[1] + C)
        memo[i] = best
        return best

    best = (0, 0)
    for i in range(n):
        L, C = dfs(i)
        if L > best[0]:
            best = (L, C)
        elif L == best[0]:
            best = (L, best[1] + C)
    sys.setrecursionlimit(old)
    return best[1]

# Bài 5
def exercise5_solution(nums):
    n = len(nums)
    f, cnt = [1] * n, [1] * n
    mx = ans = 0
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                if f[i] < f[j] + 1:
                    f[i], cnt[i] = f[j] + 1, cnt[j]
                elif f[i] == f[j] + 1:
                    cnt[i] += cnt[j]
        if f[i] > mx:
            mx, ans = f[i], cnt[i]
        elif f[i] == mx:
            ans += cnt[i]
    return mx, ans

for nums, expected in examples:
    assert exercise3_solution(nums) == expected
    assert exercise4_solution(nums) == expected
assert exercise5_solution([1, 3, 5, 4, 7]) == (4, 2)
assert exercise5_solution([2, 2, 2, 2, 2]) == (1, 5)
print("Đáp án bài 3, 4, 5 đúng trên ví dụ")

random.seed(19)
for _ in range(400):
    nums = [random.randint(-3, 3) for _ in range(random.randint(1, 9))]
    want = oracle(nums)
    assert exercise3_solution(nums) == want
    assert exercise4_solution(nums) == want
    assert exercise5_solution(nums)[1] == want
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng trên ví dụ


Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (DP hai mảng, O(n²) thời gian, O(n) bộ nhớ):**

```python
class Solution:
    def findNumberOfLIS(self, nums: List[int]) -> int:
        n = len(nums)
        f = [1] * n          # f[i]  : độ dài LIS kết thúc tại i
        cnt = [1] * n        # cnt[i]: số LIS (độ dài f[i]) kết thúc tại i
        mx = ans = 0
        for i in range(n):
            for j in range(i):
                if nums[j] < nums[i]:                 # tăng NGẶT
                    if f[i] < f[j] + 1:               # tốt hơn -> thay
                        f[i], cnt[i] = f[j] + 1, cnt[j]
                    elif f[i] == f[j] + 1:            # bằng -> cộng
                        cnt[i] += cnt[j]
            if f[i] > mx:
                mx, ans = f[i], cnt[i]
            elif f[i] == mx:
                ans += cnt[i]
        return ans
```

**Checklist trước khi submit:**

- [ ] Điều kiện nối là `nums[j] < nums[i]` (ngặt).
- [ ] `f` và `cnt` khởi tạo **toàn 1**.
- [ ] Khi độ dài tốt hơn: **thay** `cnt[i] = cnt[j]`; khi bằng: **cộng** `cnt[i] += cnt[j]`.
- [ ] Đáp án = **tổng** `cnt[i]` trên mọi `i` có `f[i]` lớn nhất (dùng `mx`, `ans` chạy cùng lúc).
- [ ] Không khử trùng mảng: các phần tử bằng nhau ở vị trí khác nhau là các dãy khác nhau.
- [ ] Đã thử: `[1]`, `[2,2,2,2,2]`, mảng tăng hẳn, giảm hẳn, `[1,2,1,2]`, `[1,3,5,4,7]`.
- [ ] Đã so với `oracle` 2^n trên mảng nhỏ ngẫu nhiên có nhiều số trùng.
- [ ] Độ phức tạp: Time O(n²) (n ≤ 2000), Space O(n).